# 06. BBS's own reports on Boro flood damage

The BBS yearbooks estimate flood damage separately from crop area. This notebook reads those
reports, because they show how flood losses were recorded in the official statistics.

It writes two tables. `data/bbs_reported_damage.csv` holds the district damage tables for the
floods of 2016, 2017 and 2022. `outputs/table_chandpur_2017.csv` holds Chandpur's rows by rice type,
which show the same area printed for HYV and hybrid Boro in 2016-17.

Both tables are kept in the repository, so this step is skipped when the yearbook PDFs are not
available. No value is corrected.

In [1]:
import re
from pathlib import Path

import pandas as pd
import pypdf

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
PDF = ROOT.parent / "bd-boro-data-framework" / "data" / "bronze" / "bbs" / "pdfs"
ACRE_HA = 0.404686
NAMES = {"Habigang": "Habiganj", "Maulbibazar": "Moulvibazar", "Maulvibazar": "Moulvibazar", "Sunamgang": "Sunamganj", "Sylhet": "Sylhet",
         "Netrokona": "Netrokona", "Kishoregong": "Kishoreganj", "Bangladesh": "Bangladesh"}
HAVE_PDFS = (PDF / "bbs_agri_yearbook_2017.pdf").exists()
print("yearbook PDFs found:", HAVE_PDFS)

yearbook PDFs found: True


## Damage tables

Each table gives the area fully and partly damaged, the total in terms of full damage, and the
production loss. The harvest year, edition, PDF page and the heading of the Total Boro block are
listed below. The headings are spelled as printed.

In [2]:
TABLES = [(2016, 2017, 367, r"Tatal Boro"), (2017, 2017, 368, r"Total Boro")]
ROW = r"^\s*([A-Za-z]+)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+(\d+)\s*$"
if HAVE_PDFS:
    rows = []
    for year, ed, page, mark in TABLES:
        t = pypdf.PdfReader(PDF / f"bbs_agri_yearbook_{ed}.pdf").pages[page - 1].extract_text()
        block = t[re.search(mark, t).start():]
        for m in re.finditer(ROW, block, re.M):
            if m.group(1) in NAMES:
                rows.append(dict(harvest_year=year, district=NAMES[m.group(1)], fully_ac=int(m.group(2)), partly_ac=int(m.group(3)),
                                 total_ac=int(m.group(6)), loss_t=int(m.group(8)), edition=ed, pdf_page=page))
    # The 2022 table has a different layout: area under crop, partly damaged, %, equivalent, fully, total, yield, loss
    t = pypdf.PdfReader(PDF / "bbs_agri_yearbook_2022.pdf").pages[441].extract_text()
    m = re.search(r"Boro\s+Netrokona\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)", t)
    for name in ("Netrokona", "Bangladesh"):
        rows.append(dict(harvest_year=2022, district=name, fully_ac=float(m.group(5)), partly_ac=float(m.group(2)),
                         total_ac=float(m.group(6)), loss_t=float(m.group(8)), edition=2022, pdf_page=442))
    d = pd.DataFrame(rows)
    d["total_ha"] = d.total_ac * ACRE_HA
    d.to_csv(ROOT / "data" / "bbs_reported_damage.csv", index=False)
d = pd.read_csv(ROOT / "data" / "bbs_reported_damage.csv")
print(d.drop(columns=["edition", "pdf_page"]).round(0).to_string())

    harvest_year     district  fully_ac  partly_ac  total_ac    loss_t  total_ha
0           2016       Sylhet    5074.0    27485.0   11350.0   12550.0    4593.0
1           2016  Moulvibazar     160.0     2810.0    1753.0    2473.0     709.0
2           2016    Sunamganj   49230.0    37203.0   57850.0   81625.0   23411.0
3           2016     Habiganj     473.0      100.0     526.0     944.0     213.0
4           2016   Bangladesh   54937.0    67598.0   71479.0   97592.0   28927.0
5           2017     Habiganj   31549.0    44032.0   53758.0   82600.0   21755.0
6           2017  Moulvibazar   19822.0    14089.0   21900.0   30834.0    8863.0
7           2017    Sunamganj  307233.0    40009.0  311022.0  457376.0  125866.0
8           2017       Sylhet   33146.0    52431.0   50315.0   57840.0   20362.0
9           2017    Netrokona   33113.0    21790.0   40957.0   66611.0   16575.0
10          2017  Kishoreganj   91120.0    15990.0   97356.0  170437.0   39399.0
11          2017   Banglades

## Chandpur's rows by rice type

Chandpur's harvested area jumps in 2017 with no flood. Its total is the sum of local, HYV and
hybrid Boro, so each is read from both editions that print the year.

In [3]:
if HAVE_PDFS:
    rows = []
    for ed, label in ((2017, "2015-16 and 2016-17"), (2018, "2016-17 and 2017-18")):
        r = pypdf.PdfReader(PDF / f"bbs_agri_yearbook_{ed}.pdf")
        for page, kind in ((89, "Local"), (91, "HYV"), (93, "Hybrid"), (95, "Total")):
            line = next(x for x in (r.pages[page - 1].extract_text() or "").splitlines() if "Chandpur" in x)
            n = re.findall(r"[\d.]+", line.split("Chandpur")[1])
            rows.append(dict(edition=ed, pdf_page=page, type=kind, first_year_acres=int(n[0]), second_year_acres=int(n[5]), years=label))
    pd.DataFrame(rows).to_csv(ROOT / "outputs" / "table_chandpur_2017.csv", index=False)
print(pd.read_csv(ROOT / "outputs" / "table_chandpur_2017.csv").to_string())

   edition  pdf_page    type  first_year_acres  second_year_acres                years
0     2017        89   Local              2263                883  2015-16 and 2016-17
1     2017        91     HYV            138933             136046  2015-16 and 2016-17
2     2017        93  Hybrid             12856             136046  2015-16 and 2016-17
3     2017        95   Total            154052             272975  2015-16 and 2016-17
4     2018        89   Local               883               1109  2016-17 and 2017-18
5     2018        91     HYV            136046             150535  2016-17 and 2017-18
6     2018        93  Hybrid            136046              15730  2016-17 and 2017-18
7     2018        95   Total            272975             167374  2016-17 and 2017-18
